# Project: GPT-1 챗봇(생성 모델) 만들기

한국어 챗봇 데이터(songys/Chatbot_data)를 이용해, 기존 **Transformer(인코더-디코더) 챗봇** 노트북의 코드를 변형하여
**GPT-1** (Radford et al., 2018, *Improving Language Understanding by Generative Pre-Training*) 모델을 구현하고 **pretrain(언어모델링)** 을 수행합니다.

| 단계 | 내용 |
|---|---|
| Step 1~4 | 데이터 다운로드 · 정제 · 토큰화 · Augmentation (기존과 동일) |
| Step 5 | **GPT 입력 형태**로 데이터 변형 (`<start> 질문 <sep> 답변 <end>` 단일 시퀀스) 및 벡터화 |
| Step 6 | **GPT-1 모델** 구성 · pretrain(다음 토큰 예측) 훈련 |
| Step 7 | 답변 생성 및 BLEU 측정 |

---

## ✅ [평가기준 1] Transformer 와 비교해 변경이 필요한 부분 (아키텍처 상 변경사항, 블럭 단위 서술)

GPT-1은 Transformer(Vaswani et al., 2017)의 **디코더만을 쌓아 만든 단방향 언어모델**입니다.



### ① 전체 구조 블럭: 인코더 제거 → 디코더 전용(decoder-only)
- Transformer: **Encoder(질문 인코딩) + Decoder(답변 생성)** 두 개의 스택.
- GPT-1: **Encoder 스택을 통째로 삭제**하고 Decoder 블럭 n개만 쌓습니다. 질문/답변 구분은 모델 구조가 아니라 **입력 시퀀스 포맷**(`<sep>` 델리미터)으로 해결합니다.

### ② 디코더 블럭: Cross-Attention(인코더-디코더 어텐션) 제거
- Transformer 디코더 층: `Masked Self-Attn → Cross-Attn(인코더 출력 참조) → FFN` 3단 구성.
- GPT-1 블럭: 인코더가 없으므로 **Cross-Attention 서브층을 제거**하여 `Masked Self-Attn → FFN` 2단 구성이 됩니다. (Masked Multi Self Attention, 논문 Figure 1 왼쪽)

### ③ 입력 블럭: 사인/코사인 위치 인코딩 → **학습되는 위치 임베딩** $W_p$
- Transformer: 고정된(학습되지 않는) sinusoidal Positional Encoding 을 더함.
- GPT-1: *"We used learned position embeddings instead of the sinusoidal version proposed in the original work."* — 위치별 벡터를 **`nn.Embedding`으로 학습**하며, 입력은 $h_0 = U W_e + W_p$ (토큰 임베딩 + 위치 임베딩) 로 구성합니다. 또한 기존 코드의 임베딩 `sqrt(d_model)` 스케일링을 제거했습니다.

### ④ FFN 블럭: 활성화 함수 ReLU → **GELU**
- 논문: *"For the activation function, we used the Gaussian Error Linear Unit (GELU)."*

### ⑤ 출력 블럭: 별도 Linear 출력층 → **토큰 임베딩과 가중치 공유(weight tying)**
- Transformer 코드: `fc_out = nn.Linear(d_model, vocab_size)` 라는 별도의 출력 행렬 사용.
- GPT-1: $P(u)=\mathrm{softmax}(h_n W_e^T)$ — **토큰 임베딩 행렬 $W_e$ 를 전치해 출력층으로 재사용**합니다.

### ⑥ 초기화 블럭: 가중치 초기화 $N(0, 0.02)$
- 논문: *"a simple weight initialization of N(0, 0.02) was sufficient"* — 모든 Linear/Embedding 가중치를 평균 0, 표준편차 0.02 정규분포로 초기화합니다.

### ⑦ 데이터/목적함수 블럭: seq2seq → **언어모델링(다음 토큰 예측) pretrain** — [평가기준 2]
- Transformer: (질문, 답변) 병렬 쌍을 인코더/디코더에 따로 입력하는 seq2seq 학습.
- GPT-1: 질문과 답변을 `<start> 질문 <sep> 답변 <end>` **하나의 연속 시퀀스**로 이어붙이고, 표준 언어모델링 목적함수 $L_1(\mathcal{U}) = \sum_i \log P(u_i \mid u_{i-k},\dots,u_{i-1};\Theta)$ (모든 위치에서 다음 토큰 예측)로 pretrain 합니다. 이번 과제는 **pretrain 데이터셋과 학습만** 고려합니다(fine-tuning 없음).

### ⑧ 최적화 블럭: Noam 스케줄 → **선형 warmup + cosine annealing**
- 논문: *"The learning rate was increased linearly from zero over the first 2000 updates and annealed to 0 using a cosine schedule."* (max lr 2.5e-4, Adam)
- 데이터 규모가 논문(BooksCorpus)보다 훨씬 작으므로 warmup 스텝 수는 축소하되 **동일한 형태의 스케줄**을 구현합니다.

### (참고) 유지한 부분
- Multi-Head **Scaled Dot-Product Attention** 연산 자체, 잔차 연결 + LayerNorm(post-LN) 배치, Dropout 규제(논문도 0.1 사용)는 원 Transformer와 동일하므로 기존 코드를 재사용합니다.
- 층수/차원(논문: 12층, d_model=768, 12 heads, d_ff=3072)은 데이터가 약 1만 쌍으로 작으므로 축소했습니다. 구조는 동일합니다.


## 준비하기: 라이브러리 설치

Colab에는 `konlpy`가 기본 설치되어 있지 않으므로 설치합니다.
  
  

- `pip install` 로 이번 프로젝트에 필요한 5개 패키지를 설치합니다.
  - `konlpy` : 한국어 형태소 분석기 모음 (여기서 `Mecab` 클래스를 사용)
  - `python-mecab-ko` : 네이티브 Mecab이 설치돼 있지 않아도 pip만으로 동작하는 **백업용 Mecab** (KoNLPy Mecab 로드 실패 시 자동 대체)
  - `gdown` : Google Drive에 올려진 사전 훈련 Word2Vec(`ko.bin`)을 내려받기 위한 도구
  - `nltk` : Step 7에서 BLEU Score 계산에 사용
  - `gensim` : Word2Vec 모델을 로드/학습하기 위한 라이브러리
- `-q` 옵션은 설치 로그를 조용히(quiet) 출력하라는 의미입니다.

In [1]:
# ------------------------------------------------------------------
# 필요한 라이브러리 설치 (Colab 환경 기준)
# ------------------------------------------------------------------
# konlpy          : 한국어 형태소 분석기 (Mecab 클래스 사용 목적)
# python-mecab-ko : KoNLPy Mecab이 동작하지 않을 때를 대비한 백업 토크나이저
# gdown           : Google Drive 파일(ko.bin) 다운로드용
# nltk            : BLEU Score 계산용 (Step 7)
# gensim          : Word2Vec 로드/학습용 (Step 4)
!pip install -q konlpy python-mecab-ko gdown nltk gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.6/579.6 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 14.2 MB/s eta 0:00:00


## 라이브러리 버전 확인


사용할 라이브러리들을 `import` 한 뒤
각 라이브러리의 `__version__` 속성을 출력해서 **어떤 버전이 설치되어 있는지 확인**합니다.


In [2]:
# ------------------------------------------------------------------
# 사용할 라이브러리를 불러오고 버전을 확인한다
# ------------------------------------------------------------------
import numpy    # 수치 연산 (벡터화된 배열 처리)
import pandas   # CSV 데이터 로드/처리
import torch    # 딥러닝 프레임워크 (Transformer 구현)
import nltk     # 자연어 처리 도구 (BLEU Score)
import gensim   # Word2Vec (Lexical Substitution)

# 각 라이브러리의 버전 출력
print(numpy.__version__)
print(pandas.__version__)
print(torch.__version__)
print(nltk.__version__)
print(gensim.__version__)

2.0.2
2.2.2
2.11.0+cu128
3.9.1
4.4.0




실험의 **재현성**을 위해 난수 시드(seed)를 고정하고, 훈련에 사용할 장치(device)를 정합니다.

- `random`, `numpy`, `torch` 세 곳의 난수 생성기를 모두 같은 시드로 고정합니다.
  (데이터 증강의 랜덤 치환, 모델 가중치 초기화, 배치 셔플 등이 매번 같은 결과가 되도록)
- `torch.cuda.is_available()` 로 GPU 사용 가능 여부를 확인하여
  가능하면 `cuda`(GPU), 아니면 `cpu`를 사용합니다.

In [3]:
# ------------------------------------------------------------------
# 재현성을 위한 시드 고정 + 훈련 장치(device) 설정
# ------------------------------------------------------------------
import random
import numpy as np

SEED = 1234                     # 모든 난수 생성기에 사용할 시드 값
random.seed(SEED)               # 파이썬 내장 random 모듈 시드 고정 (lexical_sub 에서 사용)
np.random.seed(SEED)            # numpy 난수 시드 고정
torch.manual_seed(SEED)         # PyTorch CPU 난수 시드 고정 (가중치 초기화 등)
torch.cuda.manual_seed_all(SEED)  # PyTorch GPU 난수 시드 고정 (GPU 사용 시)

# GPU가 있으면 cuda, 없으면 cpu 를 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 중인 device:", device)

사용 중인 device: cuda


## Step 1. 데이터 다운로드

[songys/Chatbot_data](https://github.com/songys/Chatbot_data) 저장소의 `ChatbotData.csv`를 다운로드합니다.



- GitHub 저장소의 raw 파일 URL을 `pandas.read_csv()` 에 직접 전달하면
  별도 다운로드 과정 없이 바로 DataFrame으로 읽어올 수 있습니다.
- 데이터는 `Q`(질문), `A`(답변), `label`(감정 분류: 0 일상, 1 부정, 2 긍정) 세 개의 열로 구성됩니다.
- `data.head()` 로 앞 5행을 출력해 데이터 모양을 눈으로 확인합니다.

In [4]:
# ------------------------------------------------------------------
# ChatbotData.csv 를 GitHub 에서 바로 읽어온다
# ------------------------------------------------------------------
import pandas as pd

# songys/Chatbot_data 저장소의 raw 파일 주소
DATA_URL = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"

# pandas 는 URL 을 직접 받아 CSV 를 읽을 수 있다
data = pd.read_csv(DATA_URL)

print("데이터 크기:", data.shape)   # (행 개수, 열 개수) → 약 11,823 x 3
data.head()                        # 앞 5행 미리보기 (Q / A / label 열 확인)

데이터 크기: (11823, 3)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0




읽어 온 데이터에서 질문 열(`Q`)과 답변 열(`A`)을 각각 파이썬 리스트로 변환하여
`questions`, `answers` 변수에 나눠 저장합니다. (`label` 열은 이번 프로젝트에서 사용하지 않습니다.)
두 리스트는 **같은 인덱스가 서로 짝을 이루는** 병렬(parallel) 데이터입니다.

In [5]:
# ------------------------------------------------------------------
# 질문(Q)과 답변(A)을 각각 questions, answers 리스트에 저장
# ------------------------------------------------------------------
questions = data["Q"].tolist()   # 질문 열 → 파이썬 리스트
answers = data["A"].tolist()     # 답변 열 → 파이썬 리스트 (questions[i] 의 답이 answers[i])

# 개수가 서로 같은지, 내용은 어떤지 확인
print("questions:", len(questions), "개")
print("answers  :", len(answers), "개")
print()
print("예시 질문:", questions[0])
print("예시 답변:", answers[0])

questions: 11823 개
answers  : 11823 개

예시 질문: 12시 땡!
예시 답변: 하루가 또 가네요.


## Step 2. 데이터 정제

아래 조건을 만족하는 `preprocess_sentence()` 함수를 구현합니다.

1. 영문자의 경우, **모두 소문자로 변환**합니다.
2. 영문자와 한글, 숫자, 그리고 주요 특수문자(`? . ! ,`)를 제외하곤 **정규식을 활용하여 모두 제거**합니다.

(문장부호 양옆 공백 추가 등은 우리가 사용할 토크나이저가 처리해 주므로 구현하지 않습니다.)
  

- `sentence.lower().strip()` : 영문 대문자를 소문자로 바꾸고 문장 양끝 공백을 제거합니다. (조건 1)
- `re.sub(r"[^a-z0-9ㄱ-ㅎㅏ-ㅣ가-힣?.!,]+", " ", sentence)` : 정규식의 `[^ ... ]` 는
  "괄호 안에 나열된 문자를 **제외한** 나머지"라는 뜻이므로,
  소문자 영문(`a-z`), 숫자(`0-9`), 한글 자모/완성형(`ㄱ-ㅎㅏ-ㅣ가-힣`), 주요 특수문자(`? . ! ,`)를
  **제외한 모든 문자를 공백 하나로 치환**합니다. (조건 2)
- 마지막으로 `\s+` (연속된 공백)를 공백 하나로 정리하고 양끝 공백을 다시 제거합니다.
- 마지막 두 줄의 `print` 로 특수문자·영문이 섞인 문장이 잘 정제되는지 테스트합니다.

In [6]:
# ------------------------------------------------------------------
# Step 2. 데이터 정제 함수 preprocess_sentence()
# ------------------------------------------------------------------
import re

def preprocess_sentence(sentence):
    """문장을 소문자화하고, 허용된 문자 외에는 정규식으로 모두 제거한다."""

    # [조건 1] 영문자는 모두 소문자로 변환 + 양끝 공백 제거
    #  - str() 은 혹시 숫자 등 문자열이 아닌 값이 들어와도 안전하게 처리하기 위함
    sentence = str(sentence).lower().strip()

    # [조건 2] 허용 문자(소문자 영문, 숫자, 한글, ? . ! ,)를 "제외한" 모든 문자를 공백으로 치환
    #  - [^...] : 대괄호 안 문자들을 제외한 나머지에 매칭
    #  - ㄱ-ㅎ / ㅏ-ㅣ : 한글 자음/모음 (ㅋㅋ, ㅠㅠ 같은 표현 보존)
    #  - 가-힣       : 한글 완성형 글자 전체
    sentence = re.sub(r"[^a-z0-9ㄱ-ㅎㅏ-ㅣ가-힣?.!,]+", " ", sentence)

    # 치환 과정에서 생긴 연속 공백을 하나로 정리하고, 양끝 공백 제거
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence

# 동작 확인: 특수문자(~, @, ^, ;)는 사라지고 영문은 소문자가 되어야 한다
print(preprocess_sentence("12시 땡! Hello~~ World@@ 안녕?"))
print(preprocess_sentence("SNS보면 나만 빼고 다 행복해보여^^;;"))

12시 땡! hello world 안녕?
sns보면 나만 빼고 다 행복해보여


## Step 3. 데이터 토큰화

토큰화에는 *KoNLPy*의 `Mecab` 클래스를 사용합니다.



- 먼저 `konlpy.tag.Mecab` 을 시도합니다. Mecab은 KoNLPy 분석기 중 가장 빠르고 정확한 편이지만,
  **네이티브 mecab 프로그램이 별도로 설치되어 있어야** 동작합니다.
- Colab에 네이티브 mecab이 없어서 로드에 실패하면 `except` 로 넘어가
  pip만으로 동작하는 `python-mecab-ko` 를 대신 사용합니다.
  이때 `MecabWrapper` 클래스로 감싸서 KoNLPy와 **동일한 `morphs()` 인터페이스**를 제공하므로
  이후 코드는 어느 쪽이 선택되든 똑같이 동작합니다.
- `morphs(문장)` 은 문장을 **형태소 단위 토큰의 리스트**로 잘라 줍니다.
  마지막 줄에서 예시 문장으로 토큰화 결과를 확인합니다.

In [7]:
# ------------------------------------------------------------------
# Step 3-1. 형태소 분석기(Mecab) 준비
#   KoNLPy Mecab 사용 → 실패 시 python-mecab-ko 로 자동 대체
# ------------------------------------------------------------------
try:
    # 1순위: KoNLPy 의 Mecab 클래스 (네이티브 mecab 필요)
    from konlpy.tag import Mecab
    mecab = Mecab()
    mecab.morphs("설치 확인용 문장입니다.")   # 실제로 동작하는지 한 번 호출해 확인
    print("KoNLPy Mecab 을 사용합니다.")
except Exception as err:
    # 2순위: 네이티브 mecab이 없으면 pip 만으로 동작하는 python-mecab-ko 사용
    print("KoNLPy Mecab 로드 실패 →", err)
    from mecab import MeCab

    class MecabWrapper:
        """konlpy.tag.Mecab 과 동일한 morphs() 인터페이스를 제공하는 래퍼 클래스"""
        def __init__(self):
            self._mecab = MeCab()          # python-mecab-ko 의 MeCab 객체

        def morphs(self, text):
            return self._mecab.morphs(text)  # 형태소 리스트 반환 (konlpy와 동일 형식)

    mecab = MecabWrapper()
    print("python-mecab-ko 를 사용합니다.")

# 토큰화 동작 확인: 문장이 형태소 단위 리스트로 나뉜다
print(mecab.morphs(preprocess_sentence("12시 땡! 지루하다, 놀러가고 싶어.")))

KoNLPy Mecab 로드 실패 → Install MeCab in order to use it: http://konlpy.org/en/latest/install/
python-mecab-ko 를 사용합니다.
['12', '시', '땡', '!', '지루', '하', '다', ',', '놀', '러', '가', '고', '싶', '어', '.']


아래 조건을 만족하는 `build_corpus()` 함수를 구현합니다.

1. **소스 문장 데이터**와 **타겟 문장 데이터**를 입력으로 받습니다.
2. 데이터를 앞서 정의한 `preprocess_sentence()` 함수로 정제하고, 토큰화합니다.
3. 토큰화는 **전달받은 토크나이즈 함수**를 사용합니다. (`mecab.morphs` 전달)
4. 토큰의 개수가 일정 길이 이상인 문장은 데이터에서 제외합니다.
5. **중복되는 문장은 제외**합니다. 소스는 소스대로, 타겟은 타겟대로 검사하며, 중복 쌍이 흐트러지지 않도록 쌍 단위로 제거합니다.



- `for src, tgt in zip(...)` 으로 질문·답변을 **쌍 단위로** 순회합니다.
  (쌍 단위로 처리해야 어떤 문장이 제외될 때 상대 문장도 함께 제외되어 병렬 관계가 유지됩니다.)
- 각 문장을 `preprocess_sentence()` 로 정제한 뒤, 인자로 전달받은 `tokenize_fn`(= `mecab.morphs`)으로 토큰화합니다.
- 토큰 개수가 `max_len`(기본 20) **이상**이면 그 쌍을 건너뜁니다. (조건 4)
- 중복 검사는 `seen_src`, `seen_tgt` 두 개의 `set` 으로 수행합니다.
  토큰들을 공백으로 이어붙인 문자열을 key로 만들어, **소스는 소스끼리 / 타겟은 타겟끼리** 이미 나온 문장인지 확인하고,
  어느 한쪽이라도 중복이면 **쌍 전체를 건너뛰어** 쌍이 흐트러지지 않게 합니다. (조건 5)
- 마지막에 구현한 함수로 `questions`/`answers` 를 토큰화하여 `que_corpus`, `ans_corpus` 에 저장합니다.

In [8]:
# ------------------------------------------------------------------
# Step 3-2. build_corpus(): 정제 + 토큰화 + 길이 필터 + 중복 제거
# ------------------------------------------------------------------
def build_corpus(src_data, tgt_data, tokenize_fn, max_len=20):
    """소스/타겟 문장을 정제·토큰화하고, 길이 초과·중복 문장을 쌍 단위로 제거한다.

    Args:
        src_data    : 소스 문장 리스트 (질문)          [조건 1]
        tgt_data    : 타겟 문장 리스트 (답변)          [조건 1]
        tokenize_fn : 토크나이즈 함수 (mecab.morphs)   [조건 3]
        max_len     : 이 값 이상 토큰을 가진 문장은 제외 [조건 4]

    Returns:
        (src_corpus, tgt_corpus) : 토큰 리스트들의 리스트 (서로 병렬)
    """
    src_corpus, tgt_corpus = [], []
    seen_src, seen_tgt = set(), set()   # 중복 검사용 집합 (소스용 / 타겟용 따로)

    # 질문-답변을 쌍 단위로 순회 → 제외할 때도 쌍으로 제외되어 병렬 관계 유지
    for src, tgt in zip(src_data, tgt_data):
        # [조건 2, 3] 정제 후 전달받은 함수로 토큰화
        src_tokens = tokenize_fn(preprocess_sentence(src))
        tgt_tokens = tokenize_fn(preprocess_sentence(tgt))

        # [조건 4] 토큰 개수가 max_len 이상인 문장이 있으면 그 쌍은 제외
        if len(src_tokens) >= max_len or len(tgt_tokens) >= max_len:
            continue

        # [조건 5] 중복 제거
        #  - 토큰들을 공백으로 이어붙인 문자열을 중복 검사 key 로 사용
        #  - 소스는 소스대로(seen_src), 타겟은 타겟대로(seen_tgt) 검사
        #  - 어느 한쪽이라도 이미 나온 문장이면 쌍 전체를 건너뛴다 (쌍 정렬 유지)
        src_key = " ".join(src_tokens)
        tgt_key = " ".join(tgt_tokens)
        if src_key in seen_src or tgt_key in seen_tgt:
            continue
        seen_src.add(src_key)
        seen_tgt.add(tgt_key)

        # 모든 검사를 통과한 쌍만 코퍼스에 추가
        src_corpus.append(src_tokens)
        tgt_corpus.append(tgt_tokens)

    return src_corpus, tgt_corpus


MAX_TOKEN_LEN = 20   # 토큰 개수가 이 값 이상인 문장은 제외

# questions / answers 를 각각 que_corpus / ans_corpus 로 토큰화하여 저장
que_corpus, ans_corpus = build_corpus(questions, answers, mecab.morphs,
                                      max_len=MAX_TOKEN_LEN)

print("토큰화 후 데이터 크기:", len(que_corpus), "쌍 (원본", len(questions), "쌍)")
print("질문 예시:", que_corpus[0])
print("답변 예시:", ans_corpus[0])

토큰화 후 데이터 크기: 7519 쌍 (원본 11823 쌍)
질문 예시: ['12', '시', '땡', '!']
답변 예시: ['하루', '가', '또', '가', '네요', '.']


## Step 4. Augmentation

데이터가 1만 개가량으로 적은 편이므로 **Lexical Substitution**(문장의 일부 단어를 의미가 비슷한 단어로 바꿔치기)으로 데이터를 늘립니다.

[Kyubyong/wordvectors](https://github.com/Kyubyong/wordvectors)에서 한국어로 사전 훈련된
Word2Vec 모델 **Korean (w)** 를 다운로드하여 `ko.bin` 파일을 사용합니다.



- `os.path.exists("ko.bin")` 으로 파일이 이미 있는지 확인하여 **중복 다운로드를 방지**합니다.
- 없다면 `gdown` 으로 Google Drive에서 `ko.zip`을 내려받고, `unzip` 으로 압축을 풀어 `ko.bin`을 얻습니다.
- 마지막 `ls -l ko*` 로 파일이 준비되었는지 확인합니다.


In [9]:
# ------------------------------------------------------------------
# Step 4-1. 사전 훈련된 한국어 Word2Vec (Korean (w) → ko.bin) 다운로드
# ------------------------------------------------------------------
import os

# ko.bin 이 아직 없을 때만 다운로드 (이미 있으면 건너뜀)
if not os.path.exists("ko.bin"):
    # Kyubyong/wordvectors 의 Korean (w) Google Drive 링크에서 ko.zip 다운로드
    !gdown "https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU" -O ko.zip
    # 압축 해제 → ko.bin 파일이 생성된다 (-o: 덮어쓰기, -q: 조용히)
    !unzip -o -q ko.zip

# ko 로 시작하는 파일 목록을 출력해 준비 상태 확인
!ls -l ko*

Downloading...
From (original): https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU
From (redirected): https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU&confirm=t&uuid=761d35dd-faec-47a4-9686-433c22f98e57
To: /content/ko.zip
100% 80.6M/80.6M [00:01<00:00, 59.4MB/s]
-rw------- 1 root root 50697568 Dec 21  2016 ko.bin
-rw------- 1 root root 85362829 Dec 21  2016 ko.tsv
-rw-r--r-- 1 root root 80596565 Nov 25  2019 ko.zip




- `Word2Vec.load("ko.bin")` 으로 사전 훈련된 모델을 로드하고,
  단어 벡터 부분인 `.wv`(KeyedVectors)만 꺼내 `wv` 변수에 담습니다.
- `ko.bin`은 예전 버전 gensim으로 저장된 모델이라, 설치된 gensim 버전에 따라 로드에 실패할 수 있습니다.
  실패하면 `except` 로 넘어가 **우리 챗봇 코퍼스(`que_corpus + ans_corpus`)로 Word2Vec을 직접 학습**해 대체합니다.
  (사전 훈련 모델보다 어휘는 적지만 챗봇 데이터에 나오는 단어는 오히려 잘 커버합니다.)
- 마지막으로 벡터 개수와, '남자'와 유사한 단어를 출력해 **모델이 잘 동작하는지 확인**합니다.

In [10]:
# ------------------------------------------------------------------
# Step 4-2. Word2Vec 로드 (실패 시 챗봇 코퍼스로 직접 학습해 대체)
# ------------------------------------------------------------------
from gensim.models import Word2Vec

wv = None   # 단어 벡터(KeyedVectors)를 담을 변수
try:
    # 사전 훈련된 한국어 Word2Vec(ko.bin) 로드 → 단어 벡터(.wv)만 사용
    wv = Word2Vec.load("ko.bin").wv
    print("사전 훈련된 Word2Vec(ko.bin) 로드 완료!")
except Exception as err:
    # gensim 버전 차이 등으로 로드가 실패하면 → 우리 코퍼스로 직접 학습
    print("ko.bin 로드 실패 →", err)
    print("대체: 챗봇 코퍼스로 Word2Vec 을 직접 학습합니다.")
    w2v_own = Word2Vec(
        sentences=que_corpus + ans_corpus,  # 학습 데이터: 토큰화된 질문+답변 전체
        vector_size=100,                    # 단어 벡터 차원
        window=5,                           # 주변 단어 윈도 크기
        min_count=2,                        # 2회 미만 등장 단어는 무시
        workers=4,                          # 학습 스레드 수
        epochs=30,                          # 데이터가 작으므로 여러 번 반복 학습
        seed=SEED,                          # 재현성
    )
    wv = w2v_own.wv

# 로드/학습된 단어 벡터 확인
print("단어 벡터 개수:", len(wv.index_to_key))
if "남자" in wv:   # '남자'라는 단어가 어휘에 있으면 유사 단어를 출력해 본다
    print("'남자'와 유사한 단어:", wv.most_similar("남자", topn=3))

ERROR:gensim.models.word2vec:Model load error. Was model saved using code from an older Gensim Version? Try loading older model using gensim-3.8.3, then re-saving, to restore compatibility with current code.


ko.bin 로드 실패 → 'Word2Vec' object has no attribute 'wv'
대체: 챗봇 코퍼스로 Word2Vec 을 직접 학습합니다.
단어 벡터 개수: 3497
'남자'와 유사한 단어: [('여자', 0.9308663010597229), ('로서', 0.6544471383094788), ('언니', 0.6240939497947693)]


`lexical_sub()` 함수로 문장의 일부 토큰을 Word2Vec 상 가장 유사한 단어로 치환합니다.

- **Augmentation된 `que_corpus` + 원본 `ans_corpus`** 가 병렬을 이루고,
- 반대로 **원본 `que_corpus` + Augmentation된 `ans_corpus`** 가 병렬을 이루도록 하여
- **전체 데이터가 원래의 3배**가 되도록 합니다.


- `lexical_sub(tokens, wv, p)` : 문장의 각 토큰을 순회하면서,
  그 토큰이 Word2Vec 어휘에 있고(`tok in wv`) 확률 `p`(기본 30%)에 당첨되면
  `wv.most_similar(tok, topn=1)` 이 돌려주는 **가장 유사한 단어로 치환**합니다.
  나머지 토큰은 그대로 둡니다. → 뜻이 비슷하지만 표현이 조금 다른 새 문장이 만들어집니다.
- 이 함수를 질문 전체/답변 전체에 적용해 `aug_que_corpus`, `aug_ans_corpus` 를 만듭니다.
- 마지막으로 세 묶음을 이어붙여 3배 데이터를 만듭니다.
  - 1묶음: **원본 질문 + 원본 답변**
  - 2묶음: **증강 질문 + 원본 답변**  (질문이 조금 달라져도 같은 답)
  - 3묶음: **원본 질문 + 증강 답변**  (같은 질문에 표현이 다른 답)
- `assert` 로 두 리스트 길이가 서로 같고 원본의 3배인지 검증하고, 증강 결과를 출력해 확인합니다.

In [11]:
# ------------------------------------------------------------------
# Step 4-3. Lexical Substitution 으로 데이터 3배 증강
# ------------------------------------------------------------------
def lexical_sub(tokens, wv, p=0.3):
    """각 토큰을 확률 p 로 Word2Vec 상 가장 유사한 단어로 치환한다.

    Args:
        tokens : 토큰(형태소) 리스트, 예) ["지루", "하", "다"]
        wv     : gensim KeyedVectors (단어 벡터)
        p      : 각 토큰을 치환할 확률 (0.3 = 30%)

    Returns:
        일부 토큰이 유사어로 바뀐 새 토큰 리스트
    """
    new_tokens = []
    for tok in tokens:
        # 토큰이 Word2Vec 어휘에 존재하고, 확률 p 에 당첨된 경우에만 치환
        if tok in wv and random.random() < p:
            try:
                # most_similar → [(유사단어, 유사도), ...] 에서 1등 단어를 사용
                new_tokens.append(wv.most_similar(tok, topn=1)[0][0])
            except Exception:
                # 혹시 유사어 검색이 실패하면 원래 토큰 유지
                new_tokens.append(tok)
        else:
            # 어휘에 없거나 확률에 당첨되지 않은 토큰은 그대로 유지
            new_tokens.append(tok)
    return new_tokens


# Augmentation 수행: 질문/답변 각각에 대해 증강 버전 생성
aug_que_corpus = [lexical_sub(q, wv) for q in que_corpus]  # 증강된 질문 (원본 답변과 짝)
aug_ans_corpus = [lexical_sub(a, wv) for a in ans_corpus]  # 증강된 답변 (원본 질문과 짝)

# 3배 데이터 구성 (병렬 관계가 흐트러지지 않도록 같은 순서로 이어붙임)
#   [원본 que + 원본 ans] + [증강 que + 원본 ans] + [원본 que + 증강 ans]
que_total = que_corpus + aug_que_corpus + que_corpus
ans_total = ans_corpus + ans_corpus + aug_ans_corpus

# 길이 검증: 두 리스트가 서로 같고, 원본의 정확히 3배여야 한다
assert len(que_total) == len(ans_total) == 3 * len(que_corpus)
print("원본:", len(que_corpus), "쌍 → 증강 후:", len(que_total), "쌍")
print()
print("원본 질문 :", " ".join(que_corpus[0]))
print("증강 질문 :", " ".join(aug_que_corpus[0]))
print("원본 답변 :", " ".join(ans_corpus[0]))
print("증강 답변 :", " ".join(aug_ans_corpus[0]))

원본: 7519 쌍 → 증강 후: 22557 쌍

원본 질문 : 12 시 땡 !
증강 질문 : 12 시 파이팅 !
원본 답변 : 하루 가 또 가 네요 .
증강 답변 : 박 가 또 가 네요 .


## Step 5. 데이터 벡터화 — GPT 입력 형태로 변형 ✅[평가기준 2]

GPT-1은 **디코더 기반의 생성(언어)모델**이므로, 소스/타겟을 나누어 인코더·디코더에 따로 넣는 대신
질문과 답변을 **하나의 연속된 시퀀스**로 이어붙여 모델에 입력합니다.

```
<start> 질문 토큰들 <sep> 답변 토큰들 <end>
```

- `<sep>` : 질문과 답변의 경계를 알려주는 **델리미터 토큰** (논문 3.3절의 delimiter token `$` 에 해당)
- `<start>` / `<end>` : 시퀀스의 시작/끝 토큰 (논문의 ⟨s⟩, ⟨e⟩ 에 해당)
- 이번 과제는 **pretrain 을 위한 데이터셋과 학습만** 고려하므로, 이 시퀀스 전체에 대해
  "다음 토큰 예측" 언어모델링을 수행합니다. (별도의 fine-tuning 없음)


In [12]:
# ------------------------------------------------------------------
# Step 5-1. GPT 입력 시퀀스 만들기: <start> Q <sep> A <end>
# ★[GPT 변경 7] (질문, 답변) 병렬 쌍 → 단일 시퀀스 (디코더 전용 생성모델용)
# ------------------------------------------------------------------
from collections import Counter

# 특수 토큰 정의
PAD, UNK, STA, SEP, END = "<pad>", "<unk>", "<start>", "<sep>", "<end>"
#  <pad>   : 배치 내 시퀀스 길이를 맞추기 위한 패딩 토큰 (인덱스 0 고정)
#  <unk>   : 단어 사전에 없는 단어를 대신하는 토큰
#  <start> : 시퀀스의 시작을 알리는 토큰
#  <sep>   : ★[GPT 변경 7] 질문과 답변 사이의 델리미터 (GPT 논문의 '$' 델리미터 역할)
#  <end>   : 시퀀스 생성의 끝을 알리는 토큰

# 기존 Transformer 코드는 "타겟(ans)에만" <start>/<end> 를 붙였지만,
# GPT 는 질문+답변을 이어붙인 시퀀스 전체를 하나의 입력으로 사용한다.
lm_corpus = [[STA] + q + [SEP] + a + [END]
             for q, a in zip(que_total, ans_total)]

print("LM 시퀀스 개수:", len(lm_corpus))
print("LM 시퀀스 예시:", lm_corpus[0])   # ['<start>', 질문..., '<sep>', 답변..., '<end>']


LM 시퀀스 개수: 22557
LM 시퀀스 예시: ['<start>', '12', '시', '땡', '!', '<sep>', '하루', '가', '또', '가', '네요', '.', '<end>']


- `Counter` 로 `lm_corpus` **전체**에 등장하는 토큰의 빈도를 셉니다.
  GPT는 애초에 인코더/디코더 구분이 없으므로 질문·답변이 자연스럽게 **하나의 단어 사전**을 공유합니다.
- 특수 토큰 5개(`<pad>, <unk>, <start>, <sep>, <end>`)를 사전 맨 앞에 배치합니다. (`<pad>`=0)
- `vectorize()` 로 모든 시퀀스를 같은 길이(`MAX_SEQ_LEN`)의 정수 배열로 만듭니다.
  - 남는 자리는 `<pad>`(0) 로 채우고, 사전에 없는 단어는 `<unk>` 로 대체합니다.
- 마지막에 **모델 입력이 정상적으로 구성되었는지** 첫 샘플을 다시 문자열로 복원(디코딩)해 눈으로 확인합니다. ✅[평가기준 3]


In [13]:
# ------------------------------------------------------------------
# Step 5-2. 단어 사전 구축 + 벡터화 → lm_train
# ★[GPT 변경 7] enc_train/dec_train 두 개 → 단일 lm_train 하나
# ------------------------------------------------------------------
# LM 시퀀스 전체 토큰의 등장 빈도를 센다
counter = Counter(tok for sent in lm_corpus for tok in sent)

# 단어 사전: 특수 토큰 5개를 맨 앞에 두고, 나머지는 빈도 높은 순서로 나열
specials = (PAD, UNK, STA, SEP, END)
vocab = list(specials) + [w for w, _ in counter.most_common() if w not in specials]
word2idx = {w: i for i, w in enumerate(vocab)}   # 단어 → 인덱스
idx2word = {i: w for w, i in word2idx.items()}   # 인덱스 → 단어 (생성 시 사용)
VOCAB_SIZE = len(vocab)
print("단어 사전 크기:", VOCAB_SIZE)


def vectorize(corpus, maxlen):
    """토큰 리스트들의 코퍼스를 (N, maxlen) 정수 배열로 변환한다.

    - 남는 자리는 0(<pad>) 으로 채운다 (np.zeros 로 초기화했으므로 자동)
    - 사전에 없는 단어는 <unk> 인덱스로 대체
    - maxlen 보다 긴 문장은 잘라낸다
    """
    arr = np.zeros((len(corpus), maxlen), dtype=np.int64)   # 0(<pad>) 으로 초기화
    for i, sent in enumerate(corpus):
        ids = [word2idx.get(tok, word2idx[UNK]) for tok in sent][:maxlen]
        arr[i, :len(ids)] = ids   # 앞부분만 실제 인덱스로 채움 (뒤는 패딩)
    return arr


# 가장 긴 LM 시퀀스에 맞춰 최대 길이 결정 (<start>/<sep>/<end> 포함)
MAX_SEQ_LEN = max(len(s) for s in lm_corpus)

# 벡터화 → 모델 pretrain 에 사용할 최종 데이터 (단일 배열!)
lm_train = vectorize(lm_corpus, MAX_SEQ_LEN)
print("lm_train:", lm_train.shape)   # (전체 데이터 수, MAX_SEQ_LEN)

# --- ✅[평가기준 3] 모델 input 이 정상적으로 구성되었는지 확인 ---
# 첫 샘플을 인덱스 → 단어로 되돌려 <start> Q <sep> A <end> 형태인지 눈으로 검증
sample = lm_train[0]
print("\n[입력 확인] 인덱스 :", sample.tolist())
print("[입력 확인] 복원   :", " ".join(idx2word[i] for i in sample if i != 0))
assert sample[0] == word2idx[STA], "시퀀스는 <start> 로 시작해야 합니다"
assert word2idx[SEP] in sample and word2idx[END] in sample, "<sep>/<end> 가 포함되어야 합니다"
print("[입력 확인] OK: <start> 질문 <sep> 답변 <end> 형태로 구성됨")


단어 사전 크기: 6188
lm_train: (22557, 40)

[입력 확인] 인덱스 : [2, 1873, 225, 2579, 126, 3, 249, 10, 144, 10, 44, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[입력 확인] 복원   : <start> 12 시 땡 ! <sep> 하루 가 또 가 네요 . <end>
[입력 확인] OK: <start> 질문 <sep> 답변 <end> 형태로 구성됨


## Step 6. GPT-1 모델 구성 및 pretrain ✅[평가기준 3, 4]

기존 노트북의 **Transformer 코드를 수정**하여 GPT-1 을 구성합니다.

- `MultiHeadAttention` : 기존 코드 **그대로 재사용** (Scaled Dot-Product Attention 은 동일)
- `PositionwiseFFN` : 활성화 함수만 **ReLU → GELU** 로 교체 ★[GPT 변경 4]
- `EncoderLayer` : **삭제** (인코더 없음) ★[GPT 변경 1]
- `DecoderLayer` → `GPT1Block` : **Cross-Attention 제거**, Masked Self-Attn + FFN 만 남김 ★[GPT 변경 2]
- `PositionalEncoding`(sinusoidal) → **학습되는 위치 임베딩** `nn.Embedding` ★[GPT 변경 3]
  - 입력 블럭: $h_0 = U W_e + W_p$ — 토큰 임베딩에 위치 임베딩을 **더해** 위치 정보를 추가
- 출력층: `fc_out` 삭제 → **토큰 임베딩 $W_e$ 와 가중치 공유** ($h_n W_e^T$) ★[GPT 변경 5]
- 가중치 초기화: $N(0, 0.02)$ ★[GPT 변경 6]
- 마스크: 디코더 전용이므로 **causal(look-ahead) + 패딩 마스크 하나만** 사용


In [14]:
# ------------------------------------------------------------------
# Step 6-1. GPT-1 모델 정의 (기존 Transformer 코드를 수정)
# ------------------------------------------------------------------
import math
import torch.nn as nn
import torch.nn.functional as F

# ★[GPT 변경 3] 기존의 PositionalEncoding(sinusoidal, 고정값) 클래스는 삭제!
#   → GPT-1 은 "학습되는" 위치 임베딩 행렬 W_p (nn.Embedding) 를 사용한다.


class MultiHeadAttention(nn.Module):
    """멀티 헤드 어텐션 (Scaled Dot-Product Attention).

    ※ 기존 Transformer 코드와 동일 — 어텐션 연산 자체는 GPT-1 에서도 그대로다.
      GPT 에서는 항상 q=k=v=x (self-attention) + causal mask 로만 호출된다.
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 은 n_heads 로 나누어떨어져야 합니다."
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads    # head 하나가 담당하는 차원 수

        # Q, K, V 를 만드는 선형 변환 + 최종 출력 선형 변환
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """(B, L, d_model) → (B, n_heads, L, d_head) 로 쪼개 head 차원을 분리"""
        B, L, _ = x.size()
        return x.view(B, L, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, q, k, v, mask=None):
        # 1) 선형 변환 후 head 분리
        q = self.split_heads(self.wq(q))    # (B, H, Lq, Dh)
        k = self.split_heads(self.wk(k))    # (B, H, Lk, Dh)
        v = self.split_heads(self.wv(v))    # (B, H, Lk, Dh)

        # 2) 어텐션 점수 = Q·K^T / sqrt(d_head)  → (B, H, Lq, Lk)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)

        # 3) 마스크 적용: True 인 위치(패딩/미래 단어)는 -inf 로 채워
        #    softmax 후 확률이 0 이 되도록 한다 (Masked Self-Attention)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        # 4) softmax 로 확률화한 뒤 V 를 가중합
        attn = F.softmax(scores, dim=-1)    # (B, H, Lq, Lk)
        out = torch.matmul(attn, v)         # (B, H, Lq, Dh)

        # 5) head 들을 다시 이어붙이고 최종 선형 변환
        B, H, L, Dh = out.size()
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.wo(out), attn


class PositionwiseFFN(nn.Module):
    """위치별 피드포워드 네트워크: 확장(d_ff) → GELU → 축소(d_model).

    ★[GPT 변경 4] 활성화 함수를 ReLU → GELU 로 교체
      (논문: "For the activation function, we used the Gaussian Error
       Linear Unit (GELU).")
    """
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 차원 확장
            nn.GELU(),                  # ★[GPT 변경 4] ReLU → GELU
            nn.Dropout(dropout),        # 과적합 방지
            nn.Linear(d_ff, d_model),   # 원래 차원으로 축소
        )

    def forward(self, x):
        return self.net(x)


# ★[GPT 변경 1] 기존의 EncoderLayer 클래스는 통째로 삭제! (GPT 는 인코더가 없다)


class GPT1Block(nn.Module):
    """GPT-1 트랜스포머 블럭 (기존 DecoderLayer 를 수정한 것).

    ★[GPT 변경 2] 기존 DecoderLayer 의 3단 구성
        Masked Self-Attn → Cross-Attn(인코더 참조) → FFN
      에서 인코더가 사라졌으므로 Cross-Attention 서브층을 "제거"하여
        Masked Self-Attn → FFN
      2단 구성이 된다. (잔차 연결 + LayerNorm(post-LN) 은 원 Transformer 와 동일)
    """
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)  # 마스크드 셀프 어텐션
        # ★[GPT 변경 2] self.cross_attn / self.norm2(cross용) 삭제됨
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # 1) Masked Self-Attention: 각 위치가 "자기 이전" 토큰들만 참조 (단방향)
        attn_out, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))   # 잔차 연결 + 정규화
        # 2) FFN  (★Cross-Attention 단계가 사라졌다)
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class GPT1(nn.Module):
    """GPT-1: 디코더 전용(decoder-only) 언어모델 (기존 Transformer 클래스를 수정).

    논문 Section 3.1:
        h_0 = U W_e + W_p
        h_l = transformer_block(h_{l-1})
        P(u) = softmax(h_n W_e^T)

    ★[GPT 변경 1] 인코더 스택(enc_layers)과 forward 의 인코더 경로 삭제,
                   입력도 (enc_in, dec_in) 두 개 → 시퀀스 x 하나만 받는다.
    """
    def __init__(self, vocab_size, n_layers, d_model, n_heads, d_ff,
                 dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        # 토큰 임베딩 행렬 W_e  (padding_idx=0 : <pad> 임베딩은 0 벡터)
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)

        # ★[GPT 변경 3] 학습되는 위치 임베딩 행렬 W_p
        #   sinusoidal 고정값 대신, 위치(0..max_len-1)마다 d_model 차원 벡터를 "학습"
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.dropout = nn.Dropout(dropout)   # 임베딩 dropout (논문 rate 0.1)

        # ★[GPT 변경 1] 인코더 없이 GPT1Block(디코더 블럭)만 n_layers 개 쌓는다
        self.blocks = nn.ModuleList(
            [GPT1Block(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])

        # ★[GPT 변경 5] 별도의 fc_out(Linear) 삭제!
        #   출력층은 forward 에서 토큰 임베딩 W_e 를 전치해 재사용한다 (weight tying)

        # ★[GPT 변경 6] 가중치 초기화 N(0, 0.02)
        #   (논문: "a simple weight initialization of N(0, 0.02) was sufficient")
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """★[GPT 변경 6] 모든 Linear/Embedding 가중치를 N(0, 0.02) 로 초기화"""
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
            # <pad> 임베딩은 다시 0 벡터로 고정
            if isinstance(module, nn.Embedding) and module.padding_idx is not None:
                with torch.no_grad():
                    module.weight[module.padding_idx].fill_(0)

    def make_mask(self, seq):
        """디코더용 마스크 하나만 사용 (True = 어텐션에서 가려지는 위치).

        - 패딩 마스크: <pad>(=0) 위치를 가림
        - causal(look-ahead) 마스크: 자기보다 "뒤에 오는" 토큰을 가림 (상삼각 행렬)
        ※ 기존 Transformer 의 make_look_ahead_mask 와 동일한 로직.
          인코더용 pad mask 는 인코더가 없으므로 필요 없다. ★[GPT 변경 1]
        """
        pad = (seq == 0).unsqueeze(1).unsqueeze(2)               # (B,1,1,L)
        L = seq.size(1)
        future = torch.triu(torch.ones(L, L, dtype=torch.bool,
                                       device=seq.device), diagonal=1)
        return pad | future                                      # broadcast → (B,1,L,L)

    def forward(self, x):
        """x: (B, L) 토큰 인덱스 시퀀스 → (B, L, vocab_size) 다음 토큰 logits"""
        B, L = x.size()
        mask = self.make_mask(x)

        # --- 입력 블럭: h_0 = U W_e + W_p  (논문 식 (2)) ---
        # ★[GPT 변경 3] 데이터에 위치 정보를 추가하는 과정:
        #   1) 위치 인덱스 [0, 1, ..., L-1] 를 만들고            → (1, L)
        #   2) 위치 임베딩 W_p 에서 해당 위치 벡터를 lookup      → (1, L, d_model)
        #   3) 토큰 임베딩 U W_e 에 "더해" 순서 정보를 주입      → (B, L, d_model)
        #   ※ 기존 코드의 sqrt(d_model) 스케일링도 GPT 에서는 사용하지 않음
        positions = torch.arange(L, device=x.device).unsqueeze(0)   # (1, L)
        h = self.tok_emb(x) + self.pos_emb(positions)
        h = self.dropout(h)

        # --- h_l = transformer_block(h_{l-1}) : GPT 블럭 통과 ---
        for block in self.blocks:
            h = block(h, mask)

        # --- 출력 블럭: P(u) = softmax(h_n W_e^T) ---
        # ★[GPT 변경 5] 토큰 임베딩 행렬을 전치해 출력층으로 재사용 (weight tying)
        logits = h @ self.tok_emb.weight.transpose(0, 1)   # (B, L, vocab_size)
        return logits


print("GPT-1 모델 정의 완료")


GPT-1 모델 정의 완료


하이퍼파라미터를 정하고 모델 객체를 만듭니다.

- 논문의 GPT-1 은 `12층, d_model=768, 12 heads, d_ff=3072` 이지만,
  데이터가 1만 쌍 규모로 작기 때문에 **구조는 그대로 두고 크기만 축소**했습니다.
- `D_FF = 4 × D_MODEL` 비율은 논문(768→3072)과 동일하게 유지했습니다.
- `DROPOUT = 0.1` : 논문과 동일한 규제 값 (residual/embedding/attention dropout 0.1)
- 모델 생성 후 **`print(model)`** 로 GPT-1 구조를 확인합니다. ✅[평가기준 4]
  - `EncoderLayer`/`cross_attn` 이 없고, `pos_emb`(학습형 위치 임베딩)와 `GPT1Block` 만 있는지 확인하세요.


In [15]:
# ------------------------------------------------------------------
# Step 6-2. 하이퍼파라미터 설정 + 모델 생성 + print(model)
# ------------------------------------------------------------------
from torch.utils.data import TensorDataset, DataLoader

# --- 모델 하이퍼파라미터 (논문 스펙을 작은 데이터에 맞춰 축소) ---
N_LAYERS = 2      # GPT 블럭 수            (논문: 12)
D_MODEL  = 256    # 임베딩/은닉 차원        (논문: 768)
N_HEADS  = 8      # 어텐션 head 수          (논문: 12)
D_FF     = 1024   # FFN 내부 확장 차원      (논문: 3072 = 4×768, 여기도 4×256)
DROPOUT  = 0.1    # 드롭아웃 비율           (논문과 동일: 0.1)

# --- 훈련 파라미터 ---
MAX_LR       = 2.5e-4   # 최대 러닝 레이트 (논문과 동일: 2.5e-4)
WARMUP_STEPS = 400      # 선형 warmup 스텝 수 (논문 2000 → 데이터가 작으므로 축소)
BATCH_SIZE   = 64       # 배치 크기 (논문: 64 — 동일!)
EPOCHS       = 10       # 전체 데이터 반복 횟수

# 모델 생성 후 GPU/CPU 로 이동
# ★[GPT 변경 1] 입력이 하나(시퀀스)뿐이므로 max_len 은 위치 임베딩 크기로 사용
model = GPT1(VOCAB_SIZE, N_LAYERS, D_MODEL, N_HEADS, D_FF,
             dropout=DROPOUT, max_len=512).to(device)

# 학습 가능한 파라미터 수 출력 (모델 크기 확인)
print("파라미터 수:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print()

# ✅[평가기준 4] GPT 모델이 정상적으로 구성되었는지 확인
print(model)


파라미터 수: 3294720

GPT1(
  (tok_emb): Embedding(6188, 256, padding_idx=0)
  (pos_emb): Embedding(512, 256)
  (dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-1): 2 x GPT1Block(
      (self_attn): MultiHeadAttention(
        (wq): Linear(in_features=256, out_features=256, bias=True)
        (wk): Linear(in_features=256, out_features=256, bias=True)
        (wv): Linear(in_features=256, out_features=256, bias=True)
        (wo): Linear(in_features=256, out_features=256, bias=True)
      )
      (ffn): PositionwiseFFN(
        (net): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=1024, out_features=256, bias=True)
        )
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    

옵티마이저·러닝 레이트 스케줄러·손실 함수·데이터 로더를 준비합니다.

- ★[GPT 변경 8] `WarmupCosineScheduler` : GPT-1 논문의 러닝 레이트 스케줄입니다.
  - 논문: *"The learning rate was increased linearly from zero over the first 2000 updates
    and annealed to 0 using a cosine schedule."*
  - `step < warmup` : 0 → `MAX_LR` 로 **선형 증가**
  - `step ≥ warmup` : cosine 곡선을 따라 0 으로 **annealing**
  - 기존 Transformer 노트북의 Noam 스케줄을 이것으로 교체했습니다.
- 옵티마이저는 논문과 동일하게 **Adam** 을 사용합니다.
- 손실 함수는 `CrossEntropyLoss(ignore_index=0)` : `<pad>` 위치는 손실에서 제외합니다.
- ★[GPT 변경 7] 데이터 로더에는 이제 `lm_train` **하나만** 들어갑니다 (enc/dec 구분 없음).


In [16]:
# ------------------------------------------------------------------
# Step 6-3. 옵티마이저 + warmup·cosine 스케줄러 + 손실 함수 + 데이터 로더
# ------------------------------------------------------------------
class WarmupCosineScheduler:
    """★[GPT 변경 8] GPT-1 논문의 러닝 레이트 스케줄 (Noam 스케줄을 교체).

    - step < warmup  : lr 이 0 → max_lr 로 선형 증가
    - step >= warmup : cosine 곡선을 따라 max_lr → 0 으로 annealing
    """
    def __init__(self, optimizer, max_lr, warmup_steps, total_steps):
        self.optimizer = optimizer
        self.max_lr = max_lr
        self.warmup = warmup_steps
        self.total = total_steps
        self.step_num = 0                 # 지금까지 진행한 전체 스텝 수

    def step(self):
        # 1) 현재 스텝의 러닝 레이트 계산
        self.step_num += 1
        if self.step_num < self.warmup:
            # 선형 warmup: 0 → max_lr
            lr = self.max_lr * self.step_num / self.warmup
        else:
            # cosine annealing: max_lr → 0
            progress = (self.step_num - self.warmup) / max(1, self.total - self.warmup)
            lr = self.max_lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
        # 2) 옵티마이저의 모든 파라미터 그룹에 lr 적용
        for group in self.optimizer.param_groups:
            group["lr"] = lr
        # 3) 실제 파라미터 업데이트 수행
        self.optimizer.step()


# ★[GPT 변경 7] 데이터셋: lm_train 하나만 사용 (enc_train/dec_train 없음)
dataset = TensorDataset(torch.from_numpy(lm_train))
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print("배치 수 / epoch:", len(loader))

# 전체 훈련 스텝 수 = 배치 수 × epoch 수 (cosine annealing 의 종점)
TOTAL_STEPS = len(loader) * EPOCHS

# 옵티마이저: 논문과 동일한 Adam (lr 은 스케줄러가 매 스텝 덮어씀)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0)
scheduler = WarmupCosineScheduler(optimizer, MAX_LR, WARMUP_STEPS, TOTAL_STEPS)

# 손실 함수: <pad>(인덱스 0) 위치는 손실 계산에서 제외
criterion = nn.CrossEntropyLoss(ignore_index=0)


배치 수 / epoch: 353


실제 pretrain(언어모델링) 훈련 루프입니다. ✅[평가기준 4 — 훈련 진행과정 프린트]

- ★[GPT 변경 7] seq2seq(teacher forcing) 대신 **언어모델링**으로 학습합니다.
  - 입력  `lm_in  = seq[:, :-1]` → `<start> q1 ... <sep> a1 ...` (마지막 토큰 제외)
  - 정답 `lm_out = seq[:, 1:]`  → `q1 ... <sep> a1 ... <end>` (한 칸 밀림)
  - 즉 **모든 위치에서 "다음 토큰"을 맞히는** 것이 목적함수입니다:
    $L_1 = \sum_i \log P(u_i \mid u_{i-k}, \dots, u_{i-1})$
- 질문 부분도 답변 부분도 모두 예측 대상입니다 (pretrain 이므로 시퀀스 전체에 대해 LM 학습).
- `<pad>` 위치는 `ignore_index=0` 으로 손실에서 제외됩니다.
- epoch 마다 평균 손실 / 토큰 정확도 / **perplexity** 를 출력합니다.


In [17]:
# ------------------------------------------------------------------
# Step 6-4. pretrain 훈련 루프 (언어모델링: 다음 토큰 예측)
# ------------------------------------------------------------------
from tqdm.auto import tqdm   # 진행률 표시 바

for epoch in range(1, EPOCHS + 1):
    model.train()                                  # 훈련 모드 (드롭아웃 활성화)
    total_loss, total_correct, total_count = 0.0, 0, 0

    for (seq_batch,) in tqdm(loader, desc=f"Epoch {epoch:2d}", leave=False):
        # 배치를 GPU/CPU 로 이동
        seq_batch = seq_batch.to(device)

        # ★[GPT 변경 7] 언어모델링: 입력과 정답을 한 칸 어긋나게 구성
        lm_in = seq_batch[:, :-1]    # <start> q1 ... <sep> a1 ...   (모델 입력)
        lm_out = seq_batch[:, 1:]    # q1 ... <sep> a1 ... <end>     (다음 토큰 정답)

        optimizer.zero_grad()                       # 이전 배치의 그래디언트 초기화
        logits = model(lm_in)                       # ★[GPT 변경 1] 입력이 하나!
                                                    #   (B, L, vocab_size)

        # 손실 계산: (B*L, vocab) vs (B*L,) 로 펼쳐서 비교 (<pad> 는 무시됨)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), lm_out.reshape(-1))
        loss.backward()                             # 역전파 (그래디언트 계산)

        # 그래디언트 클리핑: 그래디언트 폭주로 인한 발산 방지
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scheduler.step()                            # lr 갱신 + 파라미터 업데이트

        # --- 로그용 통계 집계 ---
        total_loss += loss.item() * seq_batch.size(0)
        not_pad = lm_out != 0                                   # 패딩이 아닌 위치만
        total_correct += ((logits.argmax(-1) == lm_out) & not_pad).sum().item()
        total_count += not_pad.sum().item()

    # epoch 단위 평균 손실 / 토큰 정확도 / perplexity 출력 (훈련 진행과정 확인용)
    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch:2d} | loss {avg_loss:.4f}"
          f" | acc {total_correct / total_count:.4f}"
          f" | ppl {math.exp(min(20, avg_loss)):.2f}")


Epoch  1:   0%|          | 0/353 [00:00<?, ?it/s]

Epoch  1 | loss 6.5357 | acc 0.1207 | ppl 689.30


Epoch  2:   0%|          | 0/353 [00:00<?, ?it/s]

Epoch  2 | loss 4.2925 | acc 0.3050 | ppl 73.15


Epoch  3:   0%|          | 0/353 [00:00<?, ?it/s]

...

Epoch  8 | loss 3.1517 | acc 0.4079 | ppl 23.38


Epoch  9:   0%|          | 0/353 [00:00<?, ?it/s]

Epoch  9 | loss 3.1171 | acc 0.4117 | ppl 22.58


Epoch 10:   0%|          | 0/353 [00:00<?, ?it/s]

Epoch 10 | loss 3.1038 | acc 0.4137 | ppl 22.28


— pretrain 된 GPT-1 으로 답변을 생성합니다. ✅[평가기준 5]

- `generate()` 함수 (greedy decoding):
  1. 질문 문장을 훈련 때와 **똑같은 과정**(정제 → Mecab 토큰화 → 인덱스 변환)으로 처리합니다.
  2. ★[GPT 변경 7] 프롬프트를 **`<start> 질문토큰들 <sep>`** 형태로 만듭니다.
     (훈련 데이터와 같은 포맷 — `<sep>` 뒤부터가 모델이 이어서 생성할 "답변" 자리)
  3. 매 스텝 마지막 위치의 logits 에서 argmax 로 다음 토큰을 뽑아 시퀀스 뒤에 붙입니다.
  4. `<end>` 토큰이 나오거나 최대 길이에 도달하면 생성을 멈춥니다.
- 출력 결과물의 수준과 상관없이, 입력에 따라 모델이 **정상적으로 출력을 생성하는지** 확인하는 것이 목적입니다.


In [18]:
# ------------------------------------------------------------------
# Step 6-5. 답변 생성 (greedy decoding) + 제출 양식 출력
# ------------------------------------------------------------------
@torch.no_grad()   # 생성 시에는 그래디언트 계산 불필요 (메모리/속도 절약)
def generate(sentence, model, max_new_tokens=MAX_SEQ_LEN):
    """질문 문장을 입력받아 GPT-1 이 이어서 생성한 답변을 반환한다 (greedy)."""
    model.eval()   # 평가 모드 (드롭아웃 비활성화)

    # 1) 훈련 때와 동일한 전처리: 정제 → 토큰화 → 인덱스 변환
    tokens = mecab.morphs(preprocess_sentence(sentence))

    # 2) ★[GPT 변경 7] 프롬프트 구성: <start> 질문 <sep>
    #    (인코더가 없으므로 질문도 같은 시퀀스의 앞부분으로 넣는다)
    ids = ([word2idx[STA]]
           + [word2idx.get(tok, word2idx[UNK]) for tok in tokens]
           + [word2idx[SEP]])
    seq = torch.tensor([ids], dtype=torch.long, device=device)   # (1, L)

    # 3) 한 토큰씩 생성: 마지막 위치의 argmax 를 다음 토큰으로 선택해 뒤에 붙인다
    result = []
    ban_ids = [word2idx[PAD], word2idx[UNK], word2idx[STA], word2idx[SEP]]
    for _ in range(max_new_tokens):
        if seq.size(1) >= model.max_len:         # 위치 임베딩 범위 초과 방지
            break
        logits = model(seq)[0, -1]               # ★입력 하나! 마지막 위치 logits (vocab,)
        # 내용 토큰이 아닌 특수 토큰(<pad>/<unk>/<start>/<sep>)은 생성 대상에서 제외
        # (weight tying 으로 <pad> 의 logit 이 항상 0 이므로 학습 초반에 뽑힐 수 있음)
        logits[ban_ids] = float("-inf")
        next_id = int(logits.argmax())           # 마지막 위치에서 최고 확률 토큰
        if next_id == word2idx[END]:             # <end> 가 나오면 생성 종료
            break
        result.append(idx2word[next_id])
        # 선택한 토큰을 시퀀스 뒤에 붙이고 반복 (auto-regressive)
        seq = torch.cat([seq, torch.tensor([[next_id]], dtype=torch.long,
                                           device=device)], dim=1)
    return " ".join(result)


# --- 예문에 대한 답변 생성 (제출 양식) ---
examples = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야.",
]

print("Generations (GPT-1 pretrain)")
for i, question in enumerate(examples, 1):
    print(f"> {i}. Q: {question}")
    print(f">    A: {generate(question, model)}")

print()
print("Hyperparameters")
print(f"> n_layers: {N_LAYERS}")
print(f"> d_model: {D_MODEL}")
print(f"> n_heads: {N_HEADS}")
print(f"> d_ff: {D_FF}")
print(f"> dropout: {DROPOUT}")

print()
print("Training Parameters")
print(f"> Max LR: {MAX_LR}")
print(f"> Warmup Steps: {WARMUP_STEPS}")
print(f"> Batch Size: {BATCH_SIZE}")
print(f"> Epoch At: {EPOCHS}")


Generations (GPT-1 pretrain)
> 1. Q: 지루하다, 놀러가고 싶어.
>    A: 마음 이 필요 하 겠 네요 .
> 2. Q: 오늘 일찍 일어났더니 피곤하다.
>    A: 저 도 모르 겠 어요 .
> 3. Q: 간만에 여자친구랑 데이트 하기로 했어.
>    A: 마음 이 많이 힘든가 봐요 .
> 4. Q: 집에 있는다는 소리야.
>    A: 저 도 모르 고 있 어요 .

Hyperparameters
> n_layers: 2
> d_model: 256
> n_heads: 8
> d_ff: 1024
> dropout: 0.1

Training Parameters
> Max LR: 0.00025
> Warmup Steps: 400
> Batch Size: 64
> Epoch At: 10


## Step 7. 성능 측정하기

챗봇(GPT-1)이 주어진 질문에 적절한 답변을 하는지 확인하고,
BLEU Score를 계산하는 `calculate_bleu()` 함수를 적용해 봅니다.

- `calculate_bleu(reference, candidate)` : 정답 답변 토큰과 생성 답변 토큰의 BLEU-4 를 계산합니다.
- 원본(증강 전) 데이터에서 100개를 샘플링해 평균 BLEU 를 측정합니다.
- ※ GPT-1 pretrain 만 수행한 모델이므로 점수 자체보다는 **모델이 정상 동작하는지** 확인이 목적입니다.


In [19]:
# ------------------------------------------------------------------
# Step 7-1. calculate_bleu() 구현 + 샘플 100개 평균 BLEU 측정
# ------------------------------------------------------------------
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 짧은 문장에서 3/4-gram 이 없어 0점이 되는 것을 완화하는 smoothing
smoothie = SmoothingFunction().method1

def calculate_bleu(reference, candidate):
    """BLEU Score(0~1) 를 계산한다.

    Args:
        reference : 정답 답변의 토큰 리스트
        candidate : 모델이 생성한 답변의 토큰 리스트
    """
    if len(candidate) == 0:      # 빈 답변이면 0점 (sentence_bleu 오류 방지)
        return 0.0
    # weights=(0.25,)*4 : 1~4-gram 을 균등하게 반영하는 표준 BLEU-4
    return sentence_bleu([reference], candidate,
                         weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=smoothie)


# --- 원본(증강 전) 데이터에서 무작위 샘플을 뽑아 평균 BLEU 측정 ---
N_SAMPLES = 100
sample_indices = random.sample(range(len(que_corpus)),
                               min(N_SAMPLES, len(que_corpus)))

scores = []
for idx in tqdm(sample_indices, desc="BLEU 측정"):
    question = " ".join(que_corpus[idx])     # 토큰을 다시 문장으로 (generate 입력용)
    reference = ans_corpus[idx]              # 정답 답변 토큰
    # ★[GPT 변경 7] generate 는 <sep> 뒤에 이어 생성한 답변 부분만 반환한다
    candidate = generate(question, model).split()
    scores.append(calculate_bleu(reference, candidate))

print(f"\n평균 BLEU Score ({len(scores)}개 샘플): {np.mean(scores):.4f}")


BLEU 측정:   0%|          | 0/100 [00:00<?, ?it/s]


평균 BLEU Score (100개 샘플): 0.0436




BLEU 점수만으로는 답변 품질을 판단하기 어려우므로,
방금 평가한 샘플 중 5개를 골라 **질문 / 정답 답변 / 모델 답변 / BLEU 점수**를 나란히 출력해
모델이 실제로 어떤 대답을 하는지 눈으로 직접 확인합니다.

In [20]:
# ------------------------------------------------------------------
# Step 7-2. 몇 가지 샘플을 눈으로 직접 확인
# ------------------------------------------------------------------
for idx in sample_indices[:5]:
    question = " ".join(que_corpus[idx])                    # 질문
    reference = ans_corpus[idx]                             # 정답 답변 토큰
    candidate = generate(question, model).split()           # GPT-1 답변 토큰

    print("질문     :", question)
    print("정답 답변 :", " ".join(reference))
    print("모델 답변 :", " ".join(candidate))
    print(f"BLEU     : {calculate_bleu(reference, candidate):.4f}")
    print("-" * 60)


질문     : 올해 계속 안 좋 네
정답 답변 : 삼재 인가 봐요 .
모델 답변 : 저 도 좋 아 하 세요 .
BLEU     : 0.0330
------------------------------------------------------------
질문     : 짝사랑 중 인 내 가 이해 가 안 돼 .
정답 답변 : 짝사랑 앞 에 장사 없 지요 .
모델 답변 : 사랑 하 는 사람 이 있 나 봐요 .
BLEU     : 0.0240
------------------------------------------------------------
질문     : 여친 이 오히려 섬세 하 지 못해
정답 답변 : 사람 성향 에 따라 다른 거 니 이해 해 주 세요 .
모델 답변 : 사람 을 하 는 게 좋 을 거 예요 .
BLEU     : 0.0227
------------------------------------------------------------
질문     : 난 천재 다
정답 답변 : 제 가 더 천재 예요 .
모델 답변 : 저 도 모르 고 있 어요 .
BLEU     : 0.0330
------------------------------------------------------------
질문     : 담배 너무 비 쌈
정답 답변 : 담배 피 지 마세요 .
모델 답변 : 저 도 좋 아 하 는 사람 이 에요 .
BLEU     : 0.0211
------------------------------------------------------------


## 회고

- 기존 Transformer(인코더-디코더) 챗봇 코드를 수정해 GPT-1(디코더 전용) 모델을 구성하였습니다.
- 인코더와 Cross-Attention 을 "제거"하는 것만으로 구조가 크게 단순해지고,
  대신 `<start> 질문 <sep> 답변 <end>` 라는 **입력 포맷**이 질문/답변의 역할 구분을 떠맡는다는 점이 인상적이었습니다.
- 학습형 위치 임베딩($W_p$), GELU, weight tying($W_e^T$), $N(0,0.02)$ 초기화, warmup+cosine 스케줄 등
  논문 4.1절 Model specifications 의 세부 사항을 하나씩 코드에 반영하며 GPT-1 의 설계를 구체적으로 이해할 수 있었습니다.
- pretrain 만 수행했음에도 `<sep>` 뒤에서 그럴듯한 답변이 이어져 나오는 것을 보며,
  다음 단계(supervised fine-tuning)가 왜 효과적인지 감을 잡을 수 있었습니다.
